# 04 — Training & evaluation across all split schemes — CIC-IDS-2017 — Mutual Information study

Identical protocol to the base study: 5 models × schemes `70_30`, `80_20`, `90_10`,
`70_20_10` (train/val/test) and `cv5` (5-fold stratified CV), on the top 40 features
selected by **Mutual Information**. All splits stratified with the same seed and saved to `splits/`.
Each pipeline scales and SMOTE-balances its own training portion only. A (scheme, model)
whose `metrics.json` already exists is skipped — restarts resume.

In [1]:
DATASET = "cic2017"
FT_ROOT = "ft40_mi"
TARGET_PER_CLASS = 50_000   # per-class row cap for the in-pipeline under/over sampling
MODELS_TO_RUN = None        # None = all 5; or e.g. ["svm", "knn"]
RUN_CV = True

In [2]:
import sys
from pathlib import Path

def _find_nb_common():
    for cand in [Path.cwd(), *Path.cwd().parents]:
        for c in (cand, cand / "notebooks"):
            if (c / "nb_common.py").exists():
                return c
    raise FileNotFoundError("nb_common.py not found relative to " + str(Path.cwd()))
sys.path.insert(0, str(_find_nb_common()))

import time
import numpy as np
import pandas as pd
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.model_selection import train_test_split

import nb_common as nbc
from ids import config
from ids.schema import CANONICAL_COLUMNS
from ids.models.pipelines import build_pipeline
from ids.models.registry import MODEL_NAMES
from ids.evaluation.metrics import evaluate_model
from ids.evaluation.cross_val import cross_validate_models, summarize_cv
from ids.evaluation.plots import plot_confusion_matrix

P = nbc.ft_paths(DATASET, FT_ROOT)
log = nbc.setup_logging(P.logs / "04_experiments.log", "nb04")

df = pd.read_parquet(P.data / "clean.parquet")
X, y = df[CANONICAL_COLUMNS], df["y"].to_numpy()
top40 = nbc.load_json(P.features / "top40.json")["selected"]
mapping = nbc.load_json(P.data / "label_mapping.json")
class_names = [c for c, _ in sorted(mapping.items(), key=lambda kv: kv[1])]
n_classes = len(class_names)
models_to_run = MODELS_TO_RUN or list(MODEL_NAMES)
log.info("%d rows | %d selected features | %d classes | models=%s | target_per_class=%d",
         len(X), len(top40), n_classes, models_to_run, TARGET_PER_CLASS)

2026-07-06 11:22:09,914 INFO nb04: 2520798 rows | 40 selected features | 15 classes | models=['decision_tree', 'random_forest', 'svm', 'naive_bayes', 'knn'] | target_per_class=50000


## 1. Build (or load) all split-scheme indices — stratified, fixed seed, saved to `splits/`

In [3]:
idx = np.arange(len(X))

def holdout(test_size):
    tr, te = train_test_split(idx, test_size=test_size, stratify=y,
                              random_state=config.RANDOM_STATE)
    return {"train_idx": tr, "test_idx": te}

def train_val_test():
    tr_val, te = train_test_split(idx, test_size=0.10, stratify=y,
                                  random_state=config.RANDOM_STATE)
    tr, val = train_test_split(tr_val, test_size=2/9, stratify=y[tr_val],
                               random_state=config.RANDOM_STATE)  # 2/9 of 90% = 20%
    return {"train_idx": tr, "val_idx": val, "test_idx": te}

SPLITS = {}
p_7030 = P.splits / "70_30_indices.npz"
SPLITS["70_30"] = dict(np.load(p_7030)) if p_7030.exists() else holdout(0.30)
SPLITS["80_20"] = holdout(0.20)
SPLITS["90_10"] = holdout(0.10)
SPLITS["70_20_10"] = train_val_test()

for scheme, s in SPLITS.items():
    np.savez_compressed(P.splits / f"{scheme}_indices.npz", **s)
    log.info("%s: %s", scheme,
             " / ".join(f"{k.replace('_idx','')}={len(v)}" for k, v in s.items()))

2026-07-06 11:22:13,043 INFO nb04: 70_30: train=1764558 / test=756240
2026-07-06 11:22:14,362 INFO nb04: 80_20: train=2016638 / test=504160
2026-07-06 11:22:15,684 INFO nb04: 90_10: train=2268718 / test=252080
2026-07-06 11:22:17,071 INFO nb04: 70_20_10: train=1764558 / val=504160 / test=252080


## 2. Train + evaluate every (scheme × model)

In [4]:
def run_one(scheme, name, train_idx, test_idx, val_idx=None):
    out_dir = P.runs / scheme / name
    out_dir.mkdir(parents=True, exist_ok=True)
    metrics_path = out_dir / "metrics.json"
    if metrics_path.exists():
        log.info("SKIP %s/%s — metrics.json exists", scheme, name)
        return nbc.load_json(metrics_path)

    pipeline = build_pipeline(name, top40, target_per_class=TARGET_PER_CLASS)
    t0 = time.perf_counter()
    pipeline.fit(X.iloc[train_idx], y[train_idx])
    fit_seconds = time.perf_counter() - t0
    log.info("%s/%s: fit on %d rows in %.1fs", scheme, name, len(train_idx), fit_seconds)

    record = {"dataset": DATASET, "scheme": scheme, "model": name,
              "n_train": int(len(train_idx)), "n_test": int(len(test_idx)),
              "target_per_class": TARGET_PER_CLASS, "fit_seconds": round(fit_seconds, 3)}
    evals = [("test", test_idx, "")]
    if val_idx is not None:
        record["n_val"] = int(len(val_idx))
        evals.append(("validation", val_idx, "_val"))

    for split_name, eidx, suffix in evals:
        m, _, y_pred, y_proba = evaluate_model(name, pipeline, X.iloc[eidx], y[eidx])
        cm = confusion_matrix(y[eidx], y_pred, labels=np.arange(n_classes))
        np.save(out_dir / f"confusion_matrix{suffix}.npy", cm)
        np.save(out_dir / f"predictions{suffix}.npy", y_pred)
        if y_proba is not None:
            np.save(out_dir / f"probas{suffix}.npy", y_proba.astype(np.float32))
        rep = classification_report(y[eidx], y_pred, labels=np.arange(n_classes),
                                    target_names=class_names, output_dict=True,
                                    zero_division=0)
        pd.DataFrame(rep).T.to_csv(out_dir / f"per_class{suffix}.csv")
        if split_name == "test":
            plot_confusion_matrix(f"{name} ({scheme})", cm, class_names, out_dir)
        record[split_name] = m.to_dict()
        log.info("%s/%s [%s]: acc=%.4f f1_macro=%.4f roc_auc=%s det=%.3fms",
                 scheme, name, split_name, m.accuracy, m.f1_macro,
                 "n/a" if m.roc_auc is None else round(m.roc_auc, 4),
                 m.detection_time_ms)

    nbc.save_json(record, metrics_path)
    return record

records = []
with nbc.timed(log, "all holdout schemes"):
    for scheme, s in SPLITS.items():
        with nbc.timed(log, f"scheme {scheme}"):
            for name in models_to_run:
                records.append(run_one(scheme, name, s["train_idx"], s["test_idx"],
                                       s.get("val_idx")))
pd.DataFrame([{ "scheme": r["scheme"], "model": r["model"],
                "fit_s": r["fit_seconds"], **{k: round(v, 4) if isinstance(v, float) else v
                for k, v in r["test"].items() if k not in ("name",)}} for r in records])

2026-07-06 11:22:17,088 INFO nb04: START all holdout schemes
2026-07-06 11:22:17,090 INFO nb04: START scheme 70_30
2026-07-06 11:22:35,489 INFO nb04: 70_30/decision_tree: fit on 1764558 rows in 18.4s
2026-07-06 11:22:39,386 INFO nb04: 70_30/decision_tree [test]: acc=0.9948 f1_macro=0.7695 roc_auc=0.9976 det=0.000ms


[Parallel(n_jobs=-1)]: Using backend ThreadingBackend with 8 concurrent workers.
[Parallel(n_jobs=-1)]: Done  34 tasks      | elapsed:   14.9s


2026-07-06 11:23:19,136 INFO nb04: 70_30/random_forest: fit on 1764558 rows in 39.8s


[Parallel(n_jobs=-1)]: Done 100 out of 100 | elapsed:   36.7s finished
[Parallel(n_jobs=8)]: Using backend ThreadingBackend with 8 concurrent workers.
[Parallel(n_jobs=8)]: Done  34 tasks      | elapsed:    0.8s
[Parallel(n_jobs=8)]: Done 100 out of 100 | elapsed:    1.6s finished
[Parallel(n_jobs=8)]: Using backend ThreadingBackend with 8 concurrent workers.
[Parallel(n_jobs=8)]: Done  34 tasks      | elapsed:    0.5s
[Parallel(n_jobs=8)]: Done 100 out of 100 | elapsed:    1.4s finished
[Parallel(n_jobs=8)]: Using backend ThreadingBackend with 8 concurrent workers.
[Parallel(n_jobs=8)]: Done  34 tasks      | elapsed:    0.0s
[Parallel(n_jobs=8)]: Done 100 out of 100 | elapsed:    0.0s finished
[Parallel(n_jobs=8)]: Using backend ThreadingBackend with 8 concurrent workers.
[Parallel(n_jobs=8)]: Done  34 tasks      | elapsed:    0.5s
[Parallel(n_jobs=8)]: Done 100 out of 100 | elapsed:    1.4s finished


2026-07-06 11:23:27,768 INFO nb04: 70_30/random_forest [test]: acc=0.9954 f1_macro=0.7922 roc_auc=0.9999 det=0.003ms
2026-07-06 11:23:39,068 INFO nb04: 70_30/svm: fit on 1764558 rows in 11.2s
2026-07-06 11:23:45,799 INFO nb04: 70_30/svm [test]: acc=0.6906 f1_macro=0.3291 roc_auc=0.9491 det=0.003ms
2026-07-06 11:23:49,109 INFO nb04: 70_30/naive_bayes: fit on 1764558 rows in 3.3s
2026-07-06 11:23:55,591 INFO nb04: 70_30/naive_bayes [test]: acc=0.3412 f1_macro=0.3185 roc_auc=0.9463 det=0.001ms
2026-07-06 11:23:58,507 INFO nb04: 70_30/knn: fit on 1764558 rows in 2.9s
2026-07-06 11:41:15,891 INFO nb04: 70_30/knn [test]: acc=0.9644 f1_macro=0.6536 roc_auc=0.9911 det=0.477ms
2026-07-06 11:41:15,893 INFO nb04: END   scheme 70_30 (1138.8s)
2026-07-06 11:41:15,894 INFO nb04: START scheme 80_20
2026-07-06 11:41:32,526 INFO nb04: 80_20/decision_tree: fit on 2016638 rows in 16.6s
2026-07-06 11:41:34,972 INFO nb04: 80_20/decision_tree [test]: acc=0.9950 f1_macro=0.7757 roc_auc=0.9977 det=0.001ms


...

[Parallel(n_jobs=-1)]: Done 100 out of 100 | elapsed:   36.4s finished
[Parallel(n_jobs=8)]: Using backend ThreadingBackend with 8 concurrent workers.
[Parallel(n_jobs=8)]: Done  34 tasks      | elapsed:    0.3s
[Parallel(n_jobs=8)]: Done 100 out of 100 | elapsed:    0.6s finished
[Parallel(n_jobs=8)]: Using backend ThreadingBackend with 8 concurrent workers.
[Parallel(n_jobs=8)]: Done  34 tasks      | elapsed:    0.2s
[Parallel(n_jobs=8)]: Done 100 out of 100 | elapsed:    0.5s finished
[Parallel(n_jobs=8)]: Using backend ThreadingBackend with 8 concurrent workers.
[Parallel(n_jobs=8)]: Done  34 tasks      | elapsed:    0.0s
[Parallel(n_jobs=8)]: Done 100 out of 100 | elapsed:    0.0s finished
[Parallel(n_jobs=8)]: Using backend ThreadingBackend with 8 concurrent workers.
[Parallel(n_jobs=8)]: Done  34 tasks      | elapsed:    0.2s
[Parallel(n_jobs=8)]: Done 100 out of 100 | elapsed:    0.5s finished


2026-07-06 12:02:19,958 INFO nb04: 70_20_10/random_forest [test]: acc=0.9959 f1_macro=0.7788 roc_auc=0.9999 det=0.003ms


[Parallel(n_jobs=8)]: Using backend ThreadingBackend with 8 concurrent workers.
[Parallel(n_jobs=8)]: Done  34 tasks      | elapsed:    0.5s
[Parallel(n_jobs=8)]: Done 100 out of 100 | elapsed:    1.2s finished
[Parallel(n_jobs=8)]: Using backend ThreadingBackend with 8 concurrent workers.
[Parallel(n_jobs=8)]: Done  34 tasks      | elapsed:    0.4s
[Parallel(n_jobs=8)]: Done 100 out of 100 | elapsed:    0.9s finished
[Parallel(n_jobs=8)]: Using backend ThreadingBackend with 8 concurrent workers.
[Parallel(n_jobs=8)]: Done  34 tasks      | elapsed:    0.0s
[Parallel(n_jobs=8)]: Done 100 out of 100 | elapsed:    0.0s finished
[Parallel(n_jobs=8)]: Using backend ThreadingBackend with 8 concurrent workers.
[Parallel(n_jobs=8)]: Done  34 tasks      | elapsed:    0.3s
[Parallel(n_jobs=8)]: Done 100 out of 100 | elapsed:    1.0s finished


2026-07-06 12:02:25,289 INFO nb04: 70_20_10/random_forest [validation]: acc=0.9955 f1_macro=0.8282 roc_auc=0.9999 det=0.003ms
2026-07-06 12:02:35,436 INFO nb04: 70_20_10/svm: fit on 1764558 rows in 10.1s
2026-07-06 12:02:38,030 INFO nb04: 70_20_10/svm [test]: acc=0.6902 f1_macro=0.3212 roc_auc=0.9495 det=0.001ms
2026-07-06 12:02:41,920 INFO nb04: 70_20_10/svm [validation]: acc=0.6897 f1_macro=0.3210 roc_auc=0.9485 det=0.002ms
2026-07-06 12:02:45,123 INFO nb04: 70_20_10/naive_bayes: fit on 1764558 rows in 3.2s
2026-07-06 12:02:47,776 INFO nb04: 70_20_10/naive_bayes [test]: acc=0.3510 f1_macro=0.2709 roc_auc=0.9427 det=0.001ms
2026-07-06 12:02:51,711 INFO nb04: 70_20_10/naive_bayes [validation]: acc=0.3517 f1_macro=0.3365 roc_auc=0.9423 det=0.001ms
2026-07-06 12:02:54,371 INFO nb04: 70_20_10/knn: fit on 1764558 rows in 2.7s
2026-07-06 12:08:34,486 INFO nb04: 70_20_10/knn [test]: acc=0.9664 f1_macro=0.6555 roc_auc=0.9917 det=0.446ms
2026-07-06 12:19:31,709 INFO nb04: 70_20_10/knn [validat

,scheme,model,fit_s,accuracy,precision_macro,recall_macro,f1_macro,roc_auc,detection_time_ms,n_test
0,70_30,decision_tree,18.394,0.9948,0.7101,0.9053,0.7695,0.9976,0.0004,756240
1,70_30,random_forest,39.769,0.9954,0.7644,0.8668,0.7922,0.9999,0.0030,756240
2,70_30,svm,11.220,0.6906,0.2927,0.7727,0.3291,0.9491,0.0030,756240
3,70_30,naive_bayes,3.308,0.3412,0.3444,0.6419,0.3185,0.9463,0.0007,756240
4,70_30,knn,2.913,0.9644,0.5871,0.9085,0.6536,0.9911,0.4773,756240
5,80_20,decision_tree,16.631,0.9950,0.7164,0.9144,0.7757,0.9977,0.0007,504160
6,80_20,random_forest,37.807,0.9954,0.8157,0.9006,0.8225,0.9999,0.0027,504160
7,80_20,svm,10.138,0.6915,0.3036,0.7864,0.3355,0.9501,0.0015,504160
8,80_20,naive_bayes,4.116,0.3334,0.3429,0.6376,0.3064,0.9448,0.0007,504160
9,80_20,knn,2.689,0.9646,0.5704,0.9070,0.6402,0.9913,0.4411,504160


## 3. Five-fold stratified cross-validation (scheme `cv5`)

In [5]:
cv_dir = P.runs / "cv5"
cv_dir.mkdir(parents=True, exist_ok=True)
cv_scores_path = cv_dir / "cv_scores.csv"

if not RUN_CV:
    log.info("RUN_CV=False — skipping")
elif cv_scores_path.exists():
    log.info("SKIP cv5 — cv_scores.csv exists")
    cv_summary = pd.read_csv(cv_dir / "cv_summary.csv")
else:
    with nbc.timed(log, "5-fold stratified CV, all models"):
        scores, _ = cross_validate_models(X, y, top40, model_names=models_to_run,
                                          n_splits=5, target_per_class=TARGET_PER_CLASS)
    scores.to_csv(cv_scores_path, index=False)
    cv_summary = summarize_cv(scores)
    cv_summary.to_csv(cv_dir / "cv_summary.csv", index=False)
cv_summary if RUN_CV else None

2026-07-06 12:19:31,764 INFO nb04: START 5-fold stratified CV, all models
2026-07-06 12:19:55,149 INFO ids.evaluation.cross_val: CV decision_tree fold 1/5: f1_macro=0.7611 acc=0.9943 (21.0s)
2026-07-06 12:20:14,347 INFO ids.evaluation.cross_val: CV decision_tree fold 2/5: f1_macro=0.7908 acc=0.9947 (17.2s)
2026-07-06 12:20:33,285 INFO ids.evaluation.cross_val: CV decision_tree fold 3/5: f1_macro=0.7772 acc=0.9946 (16.9s)
2026-07-06 12:20:52,470 INFO ids.evaluation.cross_val: CV decision_tree fold 4/5: f1_macro=0.7711 acc=0.9943 (17.0s)
2026-07-06 12:21:09,604 INFO ids.evaluation.cross_val: CV decision_tree fold 5/5: f1_macro=0.7763 acc=0.9946 (15.1s)


[Parallel(n_jobs=-1)]: Using backend ThreadingBackend with 8 concurrent workers.
[Parallel(n_jobs=-1)]: Done  34 tasks      | elapsed:   14.4s
[Parallel(n_jobs=-1)]: Done 100 out of 100 | elapsed:   36.6s finished
[Parallel(n_jobs=8)]: Using backend ThreadingBackend with 8 concurrent workers.
[Parallel(n_jobs=8)]: Done  34 tasks      | elapsed:    0.5s
[Parallel(n_jobs=8)]: Done 100 out of 100 | elapsed:    1.1s finished
[Parallel(n_jobs=8)]: Using backend ThreadingBackend with 8 concurrent workers.
[Parallel(n_jobs=8)]: Done  34 tasks      | elapsed:    0.3s
[Parallel(n_jobs=8)]: Done 100 out of 100 | elapsed:    0.8s finished


2026-07-06 12:21:52,374 INFO ids.evaluation.cross_val: CV random_forest fold 1/5: f1_macro=0.8203 acc=0.9947 (38.6s)


[Parallel(n_jobs=-1)]: Using backend ThreadingBackend with 8 concurrent workers.
[Parallel(n_jobs=-1)]: Done  34 tasks      | elapsed:   18.0s
[Parallel(n_jobs=-1)]: Done 100 out of 100 | elapsed:   41.0s finished
[Parallel(n_jobs=8)]: Using backend ThreadingBackend with 8 concurrent workers.
[Parallel(n_jobs=8)]: Done  34 tasks      | elapsed:    0.5s
[Parallel(n_jobs=8)]: Done 100 out of 100 | elapsed:    1.0s finished
[Parallel(n_jobs=8)]: Using backend ThreadingBackend with 8 concurrent workers.
[Parallel(n_jobs=8)]: Done  34 tasks      | elapsed:    0.3s
[Parallel(n_jobs=8)]: Done 100 out of 100 | elapsed:    0.9s finished


2026-07-06 12:22:41,367 INFO ids.evaluation.cross_val: CV random_forest fold 2/5: f1_macro=0.7954 acc=0.9946 (44.2s)


...

[Parallel(n_jobs=-1)]: Using backend ThreadingBackend with 8 concurrent workers.
[Parallel(n_jobs=-1)]: Done  34 tasks      | elapsed:   14.5s
[Parallel(n_jobs=-1)]: Done 100 out of 100 | elapsed:   36.8s finished
[Parallel(n_jobs=8)]: Using backend ThreadingBackend with 8 concurrent workers.
[Parallel(n_jobs=8)]: Done  34 tasks      | elapsed:    0.3s
[Parallel(n_jobs=8)]: Done 100 out of 100 | elapsed:    0.8s finished
[Parallel(n_jobs=8)]: Using backend ThreadingBackend with 8 concurrent workers.
[Parallel(n_jobs=8)]: Done  34 tasks      | elapsed:    0.3s
[Parallel(n_jobs=8)]: Done 100 out of 100 | elapsed:    0.8s finished


2026-07-06 12:24:09,613 INFO ids.evaluation.cross_val: CV random_forest fold 4/5: f1_macro=0.8119 acc=0.9953 (38.7s)


[Parallel(n_jobs=-1)]: Using backend ThreadingBackend with 8 concurrent workers.
[Parallel(n_jobs=-1)]: Done  34 tasks      | elapsed:   13.6s
[Parallel(n_jobs=-1)]: Done 100 out of 100 | elapsed:   33.8s finished
[Parallel(n_jobs=8)]: Using backend ThreadingBackend with 8 concurrent workers.
[Parallel(n_jobs=8)]: Done  34 tasks      | elapsed:    0.3s
[Parallel(n_jobs=8)]: Done 100 out of 100 | elapsed:    0.8s finished
[Parallel(n_jobs=8)]: Using backend ThreadingBackend with 8 concurrent workers.
[Parallel(n_jobs=8)]: Done  34 tasks      | elapsed:    0.3s
[Parallel(n_jobs=8)]: Done 100 out of 100 | elapsed:    0.8s finished


2026-07-06 12:24:48,747 INFO ids.evaluation.cross_val: CV random_forest fold 5/5: f1_macro=0.8024 acc=0.9942 (35.5s)
2026-07-06 12:25:00,185 INFO ids.evaluation.cross_val: CV svm fold 1/5: f1_macro=0.3229 acc=0.6772 (8.0s)
2026-07-06 12:25:12,137 INFO ids.evaluation.cross_val: CV svm fold 2/5: f1_macro=0.3316 acc=0.6878 (8.3s)
2026-07-06 12:25:23,711 INFO ids.evaluation.cross_val: CV svm fold 3/5: f1_macro=0.3284 acc=0.6774 (8.6s)
2026-07-06 12:25:35,384 INFO ids.evaluation.cross_val: CV svm fold 4/5: f1_macro=0.3237 acc=0.6788 (8.0s)
2026-07-06 12:25:46,976 INFO ids.evaluation.cross_val: CV svm fold 5/5: f1_macro=0.3303 acc=0.6773 (8.4s)
2026-07-06 12:25:52,533 INFO ids.evaluation.cross_val: CV naive_bayes fold 1/5: f1_macro=0.3310 acc=0.3350 (1.8s)
2026-07-06 12:25:58,782 INFO ids.evaluation.cross_val: CV naive_bayes fold 2/5: f1_macro=0.2732 acc=0.3480 (2.0s)
2026-07-06 12:26:05,540 INFO ids.evaluation.cross_val: CV naive_bayes fold 3/5: f1_macro=0.2997 acc=0.3547 (2.5s)
2026-07-06 

,model,accuracy_mean,accuracy_std,precision_macro_mean,precision_macro_std,recall_macro_mean,recall_macro_std,f1_macro_mean,f1_macro_std,roc_auc_mean,roc_auc_std
0,decision_tree,0.994486,0.000197,0.720345,0.013398,0.915716,0.022351,0.775308,0.010790,0.997781,0.000175
1,knn,0.964914,0.000730,0.577182,0.019798,0.896610,0.007141,0.643057,0.018034,0.991370,0.000176
2,naive_bayes,0.348661,0.008219,0.314880,0.031140,0.649867,0.025840,0.298523,0.025698,0.944417,0.001976
3,random_forest,0.994744,0.000414,0.794097,0.027228,0.895622,0.014589,0.810211,0.011206,0.999911,0.000017
4,svm,0.679682,0.004557,0.296572,0.003881,0.784016,0.032712,0.327357,0.003901,0.948759,0.001184


In [6]:
n_files = sum(1 for _ in P.runs.rglob("*") if _.is_file())
log.info("notebook 04 complete — %d artifact files under %s", n_files, P.runs)

2026-07-06 13:03:26,813 INFO nb04: notebook 04 complete — 142 artifact files under /Users/MAC/Projects/IDS/backend/artifacts/ft40_mi/cic2017/runs
